# RQ2-v3 Pairwise Surrogate Regret Test — CPU only

For the six frozen trajectory states, solve 91-variable pair LPs under fixed uniform marginals. Verify the Theorem-1 implementation identity, evaluate Wasserstein surrogate regret and its minimax bound, and compare uniform, width-distance, FLOPs-distance, and shuffled-SW controls. No model loading, GPU, training, accuracy, or test data is used.

In [ ]:
import os, subprocess, sys, json, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)

## Resolve the completed trajectory diagnostic and FLOPs metadata

In [ ]:
import importlib
import rq2_pairwise_surrogate_regret, rq2_quick_trajectory_diagnostic
rq2_pairwise_surrogate_regret = importlib.reload(rq2_pairwise_surrogate_regret)
rq2_quick_trajectory_diagnostic = importlib.reload(rq2_quick_trajectory_diagnostic)
QUICK_ROOT = rq2_pairwise_surrogate_regret.find_quick_trajectory_root(
    Path('/kaggle/input'), '/kaggle/working/materialized-quick-trajectory'
)
try:
    HT_ROOT = rq2_quick_trajectory_diagnostic.find_ht_development_root(
        Path('/kaggle/input'), '/kaggle/working/materialized-ht-for-regret'
    )
except FileNotFoundError:
    HT_ROOT = None
print('Quick trajectory root:', QUICK_ROOT)
print('Optional HT root for FLOPs fallback:', HT_ROOT)

## Run Theorem 1 sanity → Theorem 2 regret/bound → controls

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/rq2-pairwise-surrogate-regret')
started = time.perf_counter()
metadata = rq2_pairwise_surrogate_regret.run_pairwise_surrogate_regret(
    QUICK_ROOT, OUTPUT_DIR, ht_root=HT_ROOT, shuffle_draws=256
)
metadata['git_commit'] = GIT_COMMIT
(OUTPUT_DIR/'metadata.json').write_text(json.dumps(metadata, indent=2)+'\n')
print(f"Completed in {time.perf_counter()-started:.1f} seconds")
print(json.dumps(metadata, indent=2))

## Inspect the six-state result

In [ ]:
import pandas as pd
from IPython.display import display, Image
for name in ('pairwise_surrogate_regret.csv','pairwise_theorem_checks.csv','pairwise_control_variances.csv'):
    print('\n', name); display(pd.read_csv(OUTPUT_DIR/name))
display(Image(filename=str(OUTPUT_DIR/'pairwise_oracle_gap_captured.png')))
display(Image(filename=str(OUTPUT_DIR/'pairwise_theorem_bound.png')))
display(Image(filename=str(OUTPUT_DIR/'pairwise_control_variances.png')))

## Validate and export

In [ ]:
required = [
    'pairwise_surrogate_regret.csv','pairwise_theorem_checks.csv',
    'pairwise_control_variances.csv','pairwise_shuffled_sw_controls.csv',
    'pairwise_policy_assignments.csv','pairwise_oracle_gap_captured.png',
    'pairwise_theorem_bound.png','pairwise_control_variances.png','metadata.json',
]
missing = [name for name in required if not (OUTPUT_DIR/name).is_file() or (OUTPUT_DIR/name).stat().st_size == 0]
assert not missing, f'Missing pairwise-regret artifacts: {missing}'
saved = json.loads((OUTPUT_DIR/'metadata.json').read_text())
assert saved['theorem1_all_pass'] and saved['theorem2_all_pass']
assert saved['training_performed'] is False and saved['test_used'] is False
bundle_path = Path('/kaggle/working/rq2-pairwise-surrogate-regret.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file(): bundle.write(path, path.relative_to(OUTPUT_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**20:.1f} MiB')
bundle_path